# EvidenceLens: AI-Powered Knowledge Assistant with RAG Architecture

## Overview
EvidenceLens is a Retrieval-Augmented Generation (RAG) system that combines:
- Document ingestion and processing
- Vector embeddings for semantic search
- LLM-powered answer generation with source citations
- Interactive query interface

## Architecture Components:
1. **Document Processing**: PDF, TXT, DOCX support
2. **Chunking Strategy**: Intelligent text splitting with overlap
3. **Vector Store**: FAISS for efficient similarity search
4. **Embeddings**: Sentence Transformers for semantic understanding
5. **LLM Integration**: OpenAI GPT or local models
6. **Retrieval**: Context-aware document retrieval
7. **Generation**: Evidence-based answer synthesis

## Step 1: Install Required Dependencies

In [ ]:
# Install all required packages
!pip install -q langchain langchain-community langchain-openai
!pip install -q sentence-transformers faiss-cpu
!pip install -q pypdf python-docx tiktoken
!pip install -q chromadb openai
!pip install -q gradio ipywidgets

print("✓ All dependencies installed successfully!")

## Step 2: Import Libraries and Setup

In [ ]:
import os
import json
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from datetime import datetime

# LangChain imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    Docx2txtLoader,
    DirectoryLoader
)
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAI

# Additional utilities
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully!")

## Step 3: Configuration and Data Classes

In [ ]:
@dataclass
class EvidenceLensConfig:
    """Configuration for EvidenceLens system"""
    
    # Embedding settings
    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    embedding_device: str = "cpu"  # Use "cuda" if GPU available
    
    # Text chunking settings
    chunk_size: int = 1000
    chunk_overlap: int = 200
    
    # Retrieval settings
    top_k_documents: int = 5
    similarity_threshold: float = 0.7
    
    # LLM settings
    llm_model: str = "gpt-3.5-turbo"  # or "gpt-4"
    temperature: float = 0.3
    max_tokens: int = 1000
    
    # Storage settings
    vector_store_path: str = "./evidencelens_vectorstore"
    documents_path: str = "./documents"
    
    # API keys (set via environment variables)
    openai_api_key: Optional[str] = None


@dataclass
class RetrievalResult:
    """Structure for retrieval results"""
    content: str
    source: str
    score: float
    metadata: Dict


@dataclass
class QueryResponse:
    """Structure for query responses"""
    answer: str
    sources: List[RetrievalResult]
    confidence: float
    timestamp: str


# Initialize configuration
config = EvidenceLensConfig()

print("✓ Configuration initialized!")
print(f"  - Embedding Model: {config.embedding_model}")
print(f"  - Chunk Size: {config.chunk_size}")
print(f"  - Top K Documents: {config.top_k_documents}")

## Step 4: Document Processing Module

In [ ]:
class DocumentProcessor:
    """Handles document loading, processing, and chunking"""
    
    def __init__(self, config: EvidenceLensConfig):
        self.config = config
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=config.chunk_size,
            chunk_overlap=config.chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
        
    def load_document(self, file_path: str) -> List:
        """Load a single document based on file extension"""
        file_path = Path(file_path)
        
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")
        
        # Select appropriate loader based on extension
        if file_path.suffix.lower() == '.pdf':
            loader = PyPDFLoader(str(file_path))
        elif file_path.suffix.lower() == '.txt':
            loader = TextLoader(str(file_path))
        elif file_path.suffix.lower() in ['.docx', '.doc']:
            loader = Docx2txtLoader(str(file_path))
        else:
            raise ValueError(f"Unsupported file type: {file_path.suffix}")
        
        documents = loader.load()
        
        # Add metadata
        for doc in documents:
            doc.metadata['source'] = file_path.name
            doc.metadata['file_path'] = str(file_path)
            doc.metadata['file_type'] = file_path.suffix
            doc.metadata['loaded_at'] = datetime.now().isoformat()
        
        return documents
    
    def load_directory(self, directory_path: str) -> List:
        """Load all supported documents from a directory"""
        directory_path = Path(directory_path)
        
        if not directory_path.exists():
            raise FileNotFoundError(f"Directory not found: {directory_path}")
        
        all_documents = []
        
        # Load PDFs
        for pdf_file in directory_path.glob("**/*.pdf"):
            try:
                docs = self.load_document(pdf_file)
                all_documents.extend(docs)
                print(f"✓ Loaded: {pdf_file.name} ({len(docs)} pages)")
            except Exception as e:
                print(f"✗ Error loading {pdf_file.name}: {e}")
        
        # Load text files
        for txt_file in directory_path.glob("**/*.txt"):
            try:
                docs = self.load_document(txt_file)
                all_documents.extend(docs)
                print(f"✓ Loaded: {txt_file.name}")
            except Exception as e:
                print(f"✗ Error loading {txt_file.name}: {e}")
        
        # Load DOCX files
        for docx_file in directory_path.glob("**/*.docx"):
            try:
                docs = self.load_document(docx_file)
                all_documents.extend(docs)
                print(f"✓ Loaded: {docx_file.name}")
            except Exception as e:
                print(f"✗ Error loading {docx_file.name}: {e}")
        
        return all_documents
    
    def chunk_documents(self, documents: List) -> List:
        """Split documents into smaller chunks"""
        chunks = self.text_splitter.split_documents(documents)
        
        # Add chunk metadata
        for i, chunk in enumerate(chunks):
            chunk.metadata['chunk_id'] = i
            chunk.metadata['chunk_size'] = len(chunk.page_content)
        
        return chunks
    
    def process_documents(self, source_path: str) -> List:
        """Complete document processing pipeline"""
        print(f"\n📄 Processing documents from: {source_path}")
        print("=" * 60)
        
        # Load documents
        if Path(source_path).is_file():
            documents = self.load_document(source_path)
        else:
            documents = self.load_directory(source_path)
        
        print(f"\n📊 Loaded {len(documents)} document(s)")
        
        # Chunk documents
        chunks = self.chunk_documents(documents)
        print(f"✂️  Created {len(chunks)} chunks")
        
        # Calculate statistics
        total_chars = sum(len(chunk.page_content) for chunk in chunks)
        avg_chunk_size = total_chars / len(chunks) if chunks else 0
        
        print(f"📈 Average chunk size: {avg_chunk_size:.0f} characters")
        print(f"📦 Total content: {total_chars:,} characters")
        
        return chunks


# Test the document processor
processor = DocumentProcessor(config)
print("✓ Document processor initialized!")

## Step 5: Vector Store and Embeddings Module

In [ ]:
class VectorStoreManager:
    """Manages vector embeddings and similarity search"""
    
    def __init__(self, config: EvidenceLensConfig):
        self.config = config
        self.embeddings = None
        self.vector_store = None
        
    def initialize_embeddings(self):
        """Initialize the embedding model"""
        print(f"\n🔧 Initializing embeddings: {self.config.embedding_model}")
        
        self.embeddings = HuggingFaceEmbeddings(
            model_name=self.config.embedding_model,
            model_kwargs={'device': self.config.embedding_device},
            encode_kwargs={'normalize_embeddings': True}
        )
        
        print("✓ Embeddings initialized!")
        
    def create_vector_store(self, documents: List):
        """Create vector store from documents"""
        if self.embeddings is None:
            self.initialize_embeddings()
        
        print(f"\n🗄️  Creating vector store from {len(documents)} chunks...")
        
        self.vector_store = FAISS.from_documents(
            documents,
            self.embeddings
        )
        
        print("✓ Vector store created!")
        
    def save_vector_store(self, path: Optional[str] = None):
        """Save vector store to disk"""
        if self.vector_store is None:
            raise ValueError("No vector store to save")
        
        save_path = path or self.config.vector_store_path
        self.vector_store.save_local(save_path)
        print(f"✓ Vector store saved to: {save_path}")
        
    def load_vector_store(self, path: Optional[str] = None):
        """Load vector store from disk"""
        if self.embeddings is None:
            self.initialize_embeddings()
        
        load_path = path or self.config.vector_store_path
        
        if not Path(load_path).exists():
            raise FileNotFoundError(f"Vector store not found at: {load_path}")
        
        self.vector_store = FAISS.load_local(
            load_path,
            self.embeddings,
            allow_dangerous_deserialization=True
        )
        
        print(f"✓ Vector store loaded from: {load_path}")
        
    def search(self, query: str, k: Optional[int] = None) -> List[Tuple]:
        """Perform similarity search"""
        if self.vector_store is None:
            raise ValueError("Vector store not initialized")
        
        k = k or self.config.top_k_documents
        
        # Search with scores
        results = self.vector_store.similarity_search_with_score(query, k=k)
        
        return results
    
    def get_retriever(self, k: Optional[int] = None):
        """Get retriever for QA chain"""
        if self.vector_store is None:
            raise ValueError("Vector store not initialized")
        
        k = k or self.config.top_k_documents
        
        return self.vector_store.as_retriever(
            search_kwargs={"k": k}
        )


# Initialize vector store manager
vector_manager = VectorStoreManager(config)
print("✓ Vector store manager initialized!")

## Step 6: RAG Query Engine

In [ ]:
class RAGQueryEngine:
    """Retrieval-Augmented Generation query engine"""
    
    def __init__(self, config: EvidenceLensConfig, vector_manager: VectorStoreManager):
        self.config = config
        self.vector_manager = vector_manager
        self.llm = None
        self.qa_chain = None
        
    def initialize_llm(self, api_key: Optional[str] = None):
        """Initialize the language model"""
        # Set API key
        if api_key:
            os.environ['OPENAI_API_KEY'] = api_key
        elif self.config.openai_api_key:
            os.environ['OPENAI_API_KEY'] = self.config.openai_api_key
        
        if 'OPENAI_API_KEY' not in os.environ:
            raise ValueError("OpenAI API key not provided")
        
        print(f"\n🤖 Initializing LLM: {self.config.llm_model}")
        
        self.llm = ChatOpenAI(
            model_name=self.config.llm_model,
            temperature=self.config.temperature,
            max_tokens=self.config.max_tokens
        )
        
        print("✓ LLM initialized!")
        
    def create_qa_chain(self):
        """Create the QA chain with custom prompt"""
        if self.llm is None:
            raise ValueError("LLM not initialized. Call initialize_llm() first.")
        
        # Custom prompt template
        template = """You are EvidenceLens, an AI knowledge assistant that provides accurate, 
evidence-based answers. Use the following pieces of context to answer the question at the end.

IMPORTANT INSTRUCTIONS:
1. Base your answer ONLY on the provided context
2. If the context doesn't contain enough information, say so clearly
3. Always cite the sources you use in your answer
4. Be concise but comprehensive
5. If there are conflicting pieces of evidence, mention both perspectives

Context:
{context}

Question: {question}

Evidence-based Answer:"""
        
        PROMPT = PromptTemplate(
            template=template,
            input_variables=["context", "question"]
        )
        
        # Get retriever
        retriever = self.vector_manager.get_retriever()
        
        # Create QA chain
        self.qa_chain = RetrievalQA.from_chain_type(
            llm=self.llm,
            chain_type="stuff",
            retriever=retriever,
            return_source_documents=True,
            chain_type_kwargs={"prompt": PROMPT}
        )
        
        print("✓ QA chain created!")
        
    def query(self, question: str) -> QueryResponse:
        """Execute a query and return structured response"""
        if self.qa_chain is None:
            raise ValueError("QA chain not initialized. Call create_qa_chain() first.")
        
        print(f"\n🔍 Processing query: {question}")
        print("=" * 60)
        
        # Execute query
        result = self.qa_chain.invoke({"query": question})
        
        # Extract answer and sources
        answer = result['result']
        source_docs = result['source_documents']
        
        # Create retrieval results
        sources = []
        for i, doc in enumerate(source_docs):
            sources.append(RetrievalResult(
                content=doc.page_content,
                source=doc.metadata.get('source', 'Unknown'),
                score=1.0 - (i * 0.1),  # Approximate relevance score
                metadata=doc.metadata
            ))
        
        # Create response
        response = QueryResponse(
            answer=answer,
            sources=sources,
            confidence=min(len(source_docs) / self.config.top_k_documents, 1.0),
            timestamp=datetime.now().isoformat()
        )
        
        return response
    
    def format_response(self, response: QueryResponse) -> str:
        """Format response for display"""
        output = []
        output.append("\n" + "=" * 60)
        output.append("📝 ANSWER")
        output.append("=" * 60)
        output.append(response.answer)
        output.append("\n" + "=" * 60)
        output.append(f"📚 SOURCES ({len(response.sources)} documents)")
        output.append("=" * 60)
        
        for i, source in enumerate(response.sources, 1):
            output.append(f"\n[{i}] {source.source}")
            output.append(f"    Relevance: {source.score:.2f}")
            output.append(f"    Preview: {source.content[:150]}...")
        
        output.append("\n" + "=" * 60)
        output.append(f"⏰ Query Time: {response.timestamp}")
        output.append(f"📊 Confidence: {response.confidence:.2%}")
        output.append("=" * 60)
        
        return "\n".join(output)


print("✓ RAG Query Engine class defined!")

## Step 7: Complete EvidenceLens System

In [ ]:
class EvidenceLens:
    """Main EvidenceLens system orchestrator"""
    
    def __init__(self, config: EvidenceLensConfig):
        self.config = config
        self.processor = DocumentProcessor(config)
        self.vector_manager = VectorStoreManager(config)
        self.query_engine = RAGQueryEngine(config, self.vector_manager)
        self.initialized = False
        
    def setup(self, documents_path: str, openai_api_key: str, force_rebuild: bool = False):
        """Complete setup pipeline"""
        print("\n" + "=" * 60)
        print("🚀 EVIDENCELENS SETUP")
        print("=" * 60)
        
        # Check if vector store exists
        vector_store_exists = Path(self.config.vector_store_path).exists()
        
        if vector_store_exists and not force_rebuild:
            print("\n📦 Found existing vector store, loading...")
            self.vector_manager.load_vector_store()
        else:
            if force_rebuild:
                print("\n🔄 Force rebuild requested, recreating vector store...")
            else:
                print("\n📦 No existing vector store found, creating new one...")
            
            # Process documents
            chunks = self.processor.process_documents(documents_path)
            
            if not chunks:
                raise ValueError("No documents were processed!")
            
            # Create vector store
            self.vector_manager.create_vector_store(chunks)
            self.vector_manager.save_vector_store()
        
        # Initialize LLM and QA chain
        self.query_engine.initialize_llm(openai_api_key)
        self.query_engine.create_qa_chain()
        
        self.initialized = True
        
        print("\n" + "=" * 60)
        print("✅ EVIDENCELENS READY!")
        print("=" * 60)
        
    def ask(self, question: str, verbose: bool = True) -> QueryResponse:
        """Ask a question"""
        if not self.initialized:
            raise ValueError("System not initialized. Call setup() first.")
        
        response = self.query_engine.query(question)
        
        if verbose:
            print(self.query_engine.format_response(response))
        
        return response
    
    def search_documents(self, query: str, k: int = 5):
        """Search documents directly without LLM"""
        if not self.initialized:
            raise ValueError("System not initialized. Call setup() first.")
        
        results = self.vector_manager.search(query, k=k)
        
        print(f"\n🔍 Search Results for: '{query}'")
        print("=" * 60)
        
        for i, (doc, score) in enumerate(results, 1):
            print(f"\n[{i}] Score: {score:.4f}")
            print(f"Source: {doc.metadata.get('source', 'Unknown')}")
            print(f"Content: {doc.page_content[:200]}...")
            print("-" * 60)
        
        return results
    
    def get_stats(self):
        """Get system statistics"""
        if not self.initialized:
            return "System not initialized"
        
        stats = {
            'embedding_model': self.config.embedding_model,
            'llm_model': self.config.llm_model,
            'chunk_size': self.config.chunk_size,
            'chunk_overlap': self.config.chunk_overlap,
            'top_k': self.config.top_k_documents,
            'vector_store_path': self.config.vector_store_path,
        }
        
        print("\n📊 EvidenceLens Statistics")
        print("=" * 60)
        for key, value in stats.items():
            print(f"{key:20s}: {value}")
        print("=" * 60)
        
        return stats


print("✓ EvidenceLens class defined!")

## Step 8: Create Sample Documents (For Testing)

In [ ]:
# Create sample documents directory
os.makedirs("./documents", exist_ok=True)

# Sample document 1: AI and Machine Learning
sample_doc1 = """Artificial Intelligence and Machine Learning: A Comprehensive Overview

Introduction to AI
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are 
programmed to think and learn. The field of AI has grown exponentially since its inception in the 1950s.

Machine Learning Fundamentals
Machine Learning (ML) is a subset of AI that enables systems to automatically learn and improve from 
experience without being explicitly programmed. The three main types of machine learning are:

1. Supervised Learning: The algorithm learns from labeled training data
2. Unsupervised Learning: The algorithm finds patterns in unlabeled data
3. Reinforcement Learning: The algorithm learns through trial and error

Deep Learning
Deep Learning is a specialized form of machine learning that uses neural networks with multiple layers. 
These networks can learn hierarchical representations of data, making them particularly effective for 
tasks like image recognition, natural language processing, and speech recognition.

Applications of AI
AI applications span numerous industries:
- Healthcare: Diagnostic tools, drug discovery, personalized medicine
- Finance: Fraud detection, algorithmic trading, risk assessment
- Transportation: Autonomous vehicles, traffic optimization
- Manufacturing: Quality control, predictive maintenance
- Customer Service: Chatbots, virtual assistants

Ethical Considerations
As AI becomes more prevalent, ethical considerations become increasingly important:
- Bias in AI systems
- Privacy concerns
- Job displacement
- Transparency and explainability
- Safety and security

Future of AI
The future of AI promises continued advancement in areas such as:
- Artificial General Intelligence (AGI)
- Quantum computing integration
- Enhanced human-AI collaboration
- More sophisticated natural language understanding
"""

# Sample document 2: Climate Change
sample_doc2 = """Climate Change: Understanding the Global Challenge

Overview
Climate change refers to long-term shifts in global temperatures and weather patterns. While climate 
change has occurred naturally throughout Earth's history, current scientific consensus indicates that 
human activities are the primary driver of recent climate change.

Causes of Climate Change
The main cause of current climate change is the increase in greenhouse gases in the atmosphere:

1. Carbon Dioxide (CO2): Released through burning fossil fuels and deforestation
2. Methane (CH4): Produced by agriculture and landfills
3. Nitrous Oxide (N2O): Emitted through agricultural and industrial activities

Effects of Climate Change
Climate change impacts multiple aspects of our planet:

Environmental Effects:
- Rising global temperatures
- Melting ice caps and glaciers
- Rising sea levels
- More frequent extreme weather events
- Ocean acidification
- Biodiversity loss

Social and Economic Impacts:
- Threats to food security
- Water scarcity
- Displacement of populations
- Economic costs of disasters
- Health impacts

Mitigation Strategies
Addressing climate change requires coordinated global action:

1. Renewable Energy: Transitioning to solar, wind, and other clean energy sources
2. Energy Efficiency: Improving efficiency in buildings, transportation, and industry
3. Reforestation: Planting trees to absorb CO2
4. Carbon Capture: Technologies to remove CO2 from the atmosphere
5. Sustainable Agriculture: Reducing emissions from farming

International Cooperation
The Paris Agreement (2015) represents a landmark international accord where nations committed to 
limiting global temperature increase to well below 2°C above pre-industrial levels.

Individual Actions
While systemic change is crucial, individual actions also matter:
- Reduce energy consumption
- Use public transportation or electric vehicles
- Reduce meat consumption
- Support sustainable businesses
- Advocate for climate policies
"""

# Sample document 3: Blockchain Technology
sample_doc3 = """Blockchain Technology: Revolutionizing Digital Transactions

What is Blockchain?
Blockchain is a distributed ledger technology that maintains a continuously growing list of records, 
called blocks, which are linked and secured using cryptography. Each block contains a cryptographic 
hash of the previous block, a timestamp, and transaction data.

Key Characteristics
1. Decentralization: No single point of control
2. Transparency: All transactions are visible to network participants
3. Immutability: Once recorded, data cannot be easily altered
4. Security: Cryptographic techniques protect data integrity

How Blockchain Works
1. A transaction is requested
2. The transaction is broadcast to network nodes
3. Nodes validate the transaction
4. Validated transactions are combined into a new block
5. The new block is added to the chain
6. The transaction is complete

Types of Blockchain
- Public Blockchain: Open to anyone (e.g., Bitcoin, Ethereum)
- Private Blockchain: Restricted access (e.g., enterprise solutions)
- Consortium Blockchain: Controlled by a group of organizations

Applications Beyond Cryptocurrency
While blockchain gained fame through Bitcoin, its applications extend far beyond:

Supply Chain Management:
- Track products from origin to consumer
- Verify authenticity
- Improve transparency

Healthcare:
- Secure patient records
- Drug traceability
- Clinical trial data management

Finance:
- Cross-border payments
- Smart contracts
- Digital identity verification

Real Estate:
- Property title management
- Simplified transactions
- Reduced fraud

Challenges and Limitations
Despite its potential, blockchain faces several challenges:
- Scalability issues
- Energy consumption (especially for proof-of-work systems)
- Regulatory uncertainty
- Integration with existing systems
- User adoption barriers

Future Outlook
The future of blockchain technology looks promising with developments in:
- Improved consensus mechanisms
- Interoperability between different blockchains
- Integration with AI and IoT
- Central Bank Digital Currencies (CBDCs)
"""

# Write sample documents
with open("./documents/ai_ml_overview.txt", "w") as f:
    f.write(sample_doc1)

with open("./documents/climate_change.txt", "w") as f:
    f.write(sample_doc2)

with open("./documents/blockchain_technology.txt", "w") as f:
    f.write(sample_doc3)

print("✓ Sample documents created in ./documents/")
print("  - ai_ml_overview.txt")
print("  - climate_change.txt")
print("  - blockchain_technology.txt")

## Step 9: Initialize EvidenceLens System

In [ ]:
# Initialize the system
evidence_lens = EvidenceLens(config)

# Set your OpenAI API key here
OPENAI_API_KEY = "your-api-key-here"  # ⚠️ REPLACE THIS WITH YOUR ACTUAL API KEY

# Setup the system (this will process documents and create vector store)
evidence_lens.setup(
    documents_path="./documents",
    openai_api_key=OPENAI_API_KEY,
    force_rebuild=False  # Set to True to rebuild vector store from scratch
)

## Step 10: Query the System - Example Usage

In [ ]:
# Example 1: Ask about AI
response1 = evidence_lens.ask(
    "What are the main types of machine learning and their characteristics?"
)

In [ ]:
# Example 2: Ask about climate change
response2 = evidence_lens.ask(
    "What are the main causes of climate change and how can we mitigate it?"
)

In [ ]:
# Example 3: Ask about blockchain
response3 = evidence_lens.ask(
    "How does blockchain technology work and what are its applications beyond cryptocurrency?"
)

In [ ]:
# Example 4: Direct document search (without LLM)
search_results = evidence_lens.search_documents(
    "renewable energy solutions",
    k=3
)

## Step 11: System Statistics and Information

In [ ]:
# Get system statistics
stats = evidence_lens.get_stats()

## Step 12: Advanced Usage - Custom Queries

In [ ]:
# Interactive query loop
def interactive_query_session():
    """Run an interactive query session"""
    print("\n" + "=" * 60)
    print("🎯 EVIDENCELENS INTERACTIVE SESSION")
    print("=" * 60)
    print("Type your questions below. Type 'quit' or 'exit' to end.")
    print("=" * 60 + "\n")
    
    while True:
        try:
            question = input("\n❓ Your Question: ").strip()
            
            if question.lower() in ['quit', 'exit', 'q']:
                print("\n👋 Goodbye! Thank you for using EvidenceLens.")
                break
            
            if not question:
                print("⚠️  Please enter a question.")
                continue
            
            # Get response
            response = evidence_lens.ask(question, verbose=True)
            
        except KeyboardInterrupt:
            print("\n\n👋 Session interrupted. Goodbye!")
            break
        except Exception as e:
            print(f"\n❌ Error: {str(e)}")

# Uncomment to run interactive session
# interactive_query_session()

## Step 13: Adding New Documents to the System

In [ ]:
def add_new_documents(new_docs_path: str):
    """Add new documents to existing vector store"""
    print(f"\n📥 Adding new documents from: {new_docs_path}")
    
    # Process new documents
    processor = DocumentProcessor(config)
    new_chunks = processor.process_documents(new_docs_path)
    
    if not new_chunks:
        print("❌ No new documents found!")
        return
    
    # Add to existing vector store
    print(f"\n➕ Adding {len(new_chunks)} chunks to vector store...")
    
    # Create embeddings for new chunks
    texts = [chunk.page_content for chunk in new_chunks]
    metadatas = [chunk.metadata for chunk in new_chunks]
    
    # Add to vector store
    evidence_lens.vector_manager.vector_store.add_texts(
        texts=texts,
        metadatas=metadatas
    )
    
    # Save updated vector store
    evidence_lens.vector_manager.save_vector_store()
    
    print("✓ Documents added successfully!")

# Example: Add new documents
# add_new_documents("./path/to/new/documents")

## Step 14: Export and Analysis Tools

In [ ]:
def export_query_history(responses: List[QueryResponse], filename: str = "query_history.json"):
    """Export query history to JSON file"""
    history = []
    
    for resp in responses:
        history.append({
            'answer': resp.answer,
            'timestamp': resp.timestamp,
            'confidence': resp.confidence,
            'num_sources': len(resp.sources),
            'sources': [
                {
                    'source': s.source,
                    'score': s.score,
                    'preview': s.content[:100]
                } for s in resp.sources
            ]
        })
    
    with open(filename, 'w') as f:
        json.dump(history, f, indent=2)
    
    print(f"✓ Query history exported to {filename}")


def analyze_document_coverage():
    """Analyze which documents are being used most frequently"""
    print("\n📊 Document Coverage Analysis")
    print("=" * 60)
    print("This would track which source documents are retrieved most often.")
    print("Implement by storing retrieval statistics during queries.")
    print("=" * 60)

# Example usage
# export_query_history([response1, response2, response3])

## Step 15: Gradio Web Interface (Optional)

In [ ]:
try:
    import gradio as gr
    
    def create_web_interface():
        """Create a Gradio web interface for EvidenceLens"""
        
        def query_interface(question):
            """Process query through Gradio interface"""
            try:
                response = evidence_lens.ask(question, verbose=False)
                
                # Format response
                answer = response.answer
                
                sources_text = "\n\n**Sources:**\n"
                for i, source in enumerate(response.sources, 1):
                    sources_text += f"\n{i}. **{source.source}** (Relevance: {source.score:.2f})\n"
                    sources_text += f"   _{source.content[:150]}..._\n"
                
                full_response = answer + sources_text
                
                return full_response
                
            except Exception as e:
                return f"Error: {str(e)}"
        
        # Create interface
        interface = gr.Interface(
            fn=query_interface,
            inputs=gr.Textbox(
                lines=3,
                placeholder="Ask a question about your documents...",
                label="Your Question"
            ),
            outputs=gr.Markdown(label="Answer"),
            title="🔍 EvidenceLens - AI Knowledge Assistant",
            description="Ask questions and get evidence-based answers from your document collection.",
            examples=[
                ["What are the main types of machine learning?"],
                ["How does blockchain technology work?"],
                ["What are the causes of climate change?"]
            ],
            theme=gr.themes.Soft()
        )
        
        return interface
    
    # Launch web interface
    # Uncomment to launch:
    # web_interface = create_web_interface()
    # web_interface.launch(share=True)
    
    print("✓ Gradio interface available. Uncomment the code above to launch.")
    
except ImportError:
    print("⚠️  Gradio not installed. Web interface not available.")

## 🎉 Quick Start Guide

### Basic Usage:
```python
# 1. Initialize the system
evidence_lens = EvidenceLens(config)

# 2. Setup with your documents
evidence_lens.setup(
    documents_path="./documents",
    openai_api_key="your-api-key",
    force_rebuild=False
)

# 3. Ask questions
response = evidence_lens.ask("Your question here?")
```

### Features:
- ✅ Multi-format document support (PDF, TXT, DOCX)
- ✅ Semantic search with vector embeddings
- ✅ Context-aware answer generation
- ✅ Source citation and tracking
- ✅ Confidence scoring
- ✅ Persistent vector storage
- ✅ Interactive and programmatic interfaces
- ✅ Optional web interface with Gradio

### Customization:
Modify the `EvidenceLensConfig` class to adjust:
- Embedding models
- Chunk sizes and overlap
- Number of retrieved documents
- LLM model and parameters
- Temperature and creativity settings

### Next Steps:
1. Add your own documents to the `./documents` folder
2. Run the setup with your OpenAI API key
3. Start asking questions!
4. Export results and analyze performance
5. Extend with additional features as needed

---

**Note:** Replace `"your-api-key-here"` with your actual OpenAI API key to use the system.

For local LLM usage (without OpenAI), you can integrate models from HuggingFace or LlamaCpp instead.